In [0]:
from datetime import datetime, timezone
from uuid import uuid4

from pyspark.sql import functions as F
from pyspark.sql.types import (
    ArrayType, BooleanType, DoubleType, IntegerType, LongType, StringType,
    StructField, StructType,
)

SCHEMA = "workspace.tmdb_lakehouse"
BRONZE = f"{SCHEMA}.bronze_movie_records"
LOGS = f"{SCHEMA}.pipeline_execution_logs"
QUARANTINE = f"{SCHEMA}.pipeline_quarantine"

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {SCHEMA}")
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {BRONZE} (
    movie_id BIGINT, load_type STRING, batch_id STRING, source_file STRING,
    source_extracted_at TIMESTAMP, window_start DATE, window_end DATE,
    ingestion_date DATE, load_timestamp TIMESTAMP, raw_payload STRING
) USING DELTA
""")
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {LOGS} (
    run_id STRING, layer STRING, load_type STRING, input_parameter STRING,
    batch_id STRING, start_time TIMESTAMP, end_time TIMESTAMP, status STRING,
    rows_inserted BIGINT, rows_updated BIGINT, rows_quarantined BIGINT,
    error_details STRING, load_timestamp TIMESTAMP, rows_processed BIGINT
) USING DELTA
""")
if "rows_processed" not in spark.table(LOGS).columns:
    spark.sql(f"ALTER TABLE {LOGS} ADD COLUMNS (rows_processed BIGINT)")
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {QUARANTINE} (
    quarantine_id STRING, layer STRING, batch_id STRING, source_file STRING,
    source_extracted_at TIMESTAMP, record_key STRING, reason STRING,
    record_hash STRING, quarantined_at TIMESTAMP
) USING DELTA
""")

In [0]:
LOG_SCHEMA = (
    "run_id STRING, layer STRING, load_type STRING, input_parameter STRING, "
    "batch_id STRING, start_time TIMESTAMP, end_time TIMESTAMP, status STRING, "
    "rows_inserted BIGINT, rows_updated BIGINT, rows_quarantined BIGINT, "
    "error_details STRING, load_timestamp TIMESTAMP, rows_processed BIGINT"
)


def utc_now():
    return datetime.now(timezone.utc).replace(tzinfo=None)


def log_run(layer, load_type, step, batch_id, started_at, status, counts=None, error=None):
    counts = counts or {}
    ended_at = utc_now()
    row = [(
        str(uuid4()), layer, load_type, step, batch_id, started_at, ended_at, status,
        counts.get("inserted", 0), counts.get("updated", 0), counts.get("quarantined", 0),
        error, ended_at, counts.get("processed", 0),
    )]
    spark.createDataFrame(row, LOG_SCHEMA).write.format("delta").mode("append").saveAsTable(LOGS)


def run_logged(layer, load_type, step, batch_id, work):
    started_at = utc_now()
    try:
        counts = work()
    except Exception as exc:
        try:
            log_run(layer, load_type, step, batch_id, started_at, "FAILURE", error=str(exc)[:4000])
        except Exception as log_exc:
            print(f"Could not write the failure log: {log_exc}")
        raise
    log_run(layer, load_type, step, batch_id, started_at, "SUCCESS", counts)
    return counts


def table_version(table):
    return spark.sql(f"DESCRIBE HISTORY {table} LIMIT 1").first()["version"]


def merge_metrics(table, since):
    last = spark.sql(f"DESCRIBE HISTORY {table} LIMIT 1").first()
    is_new_merge = last["version"] > since and last["operation"] == "MERGE"
    metrics = (last["operationMetrics"] or {}) if is_new_merge else {}
    return {
        "inserted": int(metrics.get("numTargetRowsInserted", 0)),
        "updated": int(metrics.get("numTargetRowsUpdated", 0)),
        "deleted": int(metrics.get("numTargetRowsDeleted", 0)),
    }


def sql_list(values):
    return ", ".join(str(v) for v in values)


def upsert(table, df, keys, delete_where=None, insert_only=False):
    columns = df.columns
    attrs = [c for c in columns if c not in keys and c != "load_timestamp"]
    view = f"_src_{uuid4().hex}"
    df.createOrReplaceTempView(view)

    sets = [f"t.`{c}` = s.`{c}`" for c in attrs]
    if "load_timestamp" in columns:
        keep_old = "row_hash" in columns
        sets.append(
            "t.`load_timestamp` = "
            + ("CASE WHEN NOT (t.`row_hash` <=> s.`row_hash`) THEN s.`load_timestamp` "
               "ELSE t.`load_timestamp` END" if keep_old else "s.`load_timestamp`")
        )
    clauses = []
    if attrs and not insert_only:
        changed = " OR ".join(f"NOT (t.`{c}` <=> s.`{c}`)" for c in attrs)
        clauses.append(f"WHEN MATCHED AND ({changed}) THEN UPDATE SET {', '.join(sets)}")
    names = ", ".join(f"`{c}`" for c in columns)
    values = ", ".join(f"s.`{c}`" for c in columns)
    clauses.append(f"WHEN NOT MATCHED THEN INSERT ({names}) VALUES ({values})")
    if delete_where:
        clauses.append(f"WHEN NOT MATCHED BY SOURCE AND {delete_where} THEN DELETE")

    processed = df.count()
    on = " AND ".join(f"t.`{k}` = s.`{k}`" for k in keys)
    before = table_version(table)
    spark.sql(f"MERGE INTO {table} t USING {view} s ON {on} {' '.join(clauses)}")
    return {"processed": processed, **merge_metrics(table, before)}

In [0]:
NAMED_REF = StructType([StructField("id", LongType()), StructField("name", StringType())])

MOVIE_SCHEMA = StructType([
    StructField("id", LongType()),
    StructField("imdb_id", StringType()),
    StructField("title", StringType()),
    StructField("original_title", StringType()),
    StructField("overview", StringType()),
    StructField("original_language", StringType()),
    StructField("release_date", StringType()),
    StructField("runtime", IntegerType()),
    StructField("status", StringType()),
    StructField("adult", BooleanType()),
    StructField("video", BooleanType()),
    StructField("popularity", DoubleType()),
    StructField("vote_average", DoubleType()),
    StructField("vote_count", LongType()),
    StructField("budget", LongType()),
    StructField("revenue", LongType()),
    StructField("genres", ArrayType(NAMED_REF)),
    StructField("production_companies", ArrayType(StructType([
        StructField("id", LongType()),
        StructField("name", StringType()),
        StructField("origin_country", StringType()),
    ]))),
    StructField("credits", StructType([StructField("cast", ArrayType(StructType([
        StructField("id", LongType()),
        StructField("credit_id", StringType()),
        StructField("name", StringType()),
        StructField("character", StringType()),
        StructField("order", IntegerType()),
    ])))])),
    StructField("keywords", StructType([StructField("keywords", ArrayType(NAMED_REF))])),
    StructField("reviews", StructType([StructField("results", ArrayType(StructType([
        StructField("id", StringType()),
        StructField("author", StringType()),
        StructField("author_details", StructType([
            StructField("name", StringType()),
            StructField("username", StringType()),
            StructField("avatar_path", StringType()),
            StructField("rating", DoubleType()),
        ])),
        StructField("content", StringType()),
        StructField("created_at", StringType()),
        StructField("updated_at", StringType()),
    ])))])),
])


def with_validation_reason(df):
    return df.withColumn(
        "validation_reason",
        F.when(F.col("id").isNull(), F.lit("Parsed movie ID is missing"))
        .when(
            F.col("bronze_movie_id").isNull() | (F.col("id") != F.col("bronze_movie_id")),
            F.lit("Parsed movie ID does not match Bronze movie ID"),
        )
        .when(
            F.col("title").isNull() | (F.length(F.trim("title")) == 0),
            F.lit("Movie title is missing or blank"),
        ),
    )


def merge_quarantine(df):
    key = F.coalesce(
        F.col("bronze_movie_id").cast("string"), F.col("id").cast("string"), F.lit("unknown")
    )
    record_hash = F.sha2(F.to_json(F.struct("bronze_movie_id", "id", "title")), 256)
    quarantine_id = F.sha2(
        F.concat_ws(
            "||", F.lit("Bronze-to-Silver"), F.coalesce("batch_id", F.lit("")),
            F.coalesce("source_file", F.lit("")), key, record_hash,
        ),
        256,
    )
    rows = df.select(
        quarantine_id.alias("quarantine_id"),
        F.lit("Bronze-to-Silver").alias("layer"),
        "batch_id", "source_file", "source_extracted_at",
        key.alias("record_key"),
        F.col("validation_reason").alias("reason"),
        record_hash.alias("record_hash"),
        F.current_timestamp().alias("quarantined_at"),
    )
    return upsert(QUARANTINE, rows, ["quarantine_id"], insert_only=True)